## Задание

- [x] Выбрать датасет
- [x] Определить задачу аппроксимации
- [x] Разбить датасет на обучающую и экспериментальную выборку
- [x] Провести корреляционный анализ
- [x] Выделить 1-2 переменные, которые влияют на выход
- [x] Разбить переменные на термы (распределить равномерно по графику)
- [x] Реализовать 4 функции принадлежности и для каждой построить графики
- [ ] Реализовать машину нечёткого вывода (Синглтон / Мамдани / Такаги-Сугено)

## Данные

В качестве датасета возьмём данные об использовании Инстаграм\*: [Social Media User Analysis](https://www.kaggle.com/datasets/rockyt07/social-media-user-analysis/data).
<br/>Будем аппроксимировать уровень счастья пользователя на основе его активности в соцсети.

Разделим датасет на обучающую и экспериментальную выборку в соотношении 70 к 30.

*\* соцсеть Инстаграм признана экстремистской, её деятельность запрещена на территории РФ*

## Корреляционная матрица

Начнём с вычисления корреляции между параметрами. Выберем 3 наиболее коррелирующих с параметром оценки счастья пользователя.

In [1]:
%use dataframe
%use kandy

// read dataset
val rawData = DataFrame.read("data/instagram_usage_lifestyle.csv")
val learnData = rawData.head((rawData.rowsCount() * 0.7).toInt())
val experimentData = rawData.tail((rawData.rowsCount() * 0.3).toInt())
println("""
    Rows total: ${rawData.rowsCount()}
    Rows for learning: ${learnData.rowsCount()}
    Rows for experimental: ${experimentData.rowsCount()}
    """.trimIndent())

// evaluate correlation matrix (using pearson correlation coefficient)
val correlationMatrix = learnData.select { it.all() }.corr()
DISPLAY(correlationMatrix)

// select 3 most correlated params for user happiness
val correlatedColumns: List<String> =
    correlationMatrix.first {
        it["column"] == "self_reported_happiness"
    }.let { happinessRow ->
        (happinessRow.columnNames() - "column" - "self_reported_happiness")
            .sortedBy { columnName ->
                (happinessRow[columnName] as Double).absoluteValue
            }.reversed().subList(0, 3)
    }

// show filtered correlation matrix
correlationMatrix.filter {
    it["column"] == "self_reported_happiness"
}.select("column", *correlatedColumns.toTypedArray())

Rows total: 1547896
Rows for learning: 1083527
Rows for experimental: 464368


column,user_id,age,exercise_hours_per_week,sleep_hours_per_night,perceived_stress_score,self_reported_happiness,body_mass_index,blood_pressure_systolic,blood_pressure_diastolic,daily_steps_count,weekly_work_hours,hobbies_count,social_events_per_month,books_read_per_year,volunteer_hours_per_month,travel_frequency_per_year,daily_active_minutes_instagram,sessions_per_day,posts_created_per_week,reels_watched_per_day,stories_viewed_per_day,likes_given_per_day,comments_written_per_day,dms_sent_per_week,dms_received_per_week,ads_viewed_per_day,ads_clicked_per_day,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,followers_count,following_count,notification_response_rate,account_creation_year,average_session_length_minutes,linked_accounts_count,user_engagement_score
user_id,"1,000000","-0,001529","-0,000848","0,000513","-0,000993","0,001430","0,000295","0,000217","-0,001184","-0,001283","0,000087","0,000758","0,000385","0,000885","0,000533","-0,000989","-0,001145","-0,001417","0,000405","-0,000464","-0,001399","-0,001084","-0,001318","-0,000643","-0,000812","-0,001573","-0,000874","-0,001157","-0,001203","-0,000565","-0,001605","0,001473","0,001286","0,000895","-0,001458","-0,000795","0,000191","0,001772"
age,"-0,001529","1,000000","-0,001245","0,001742","-0,000561","0,000046","0,001240","-0,000324","0,000152","0,000573","-0,000597","0,000163","0,000967","-0,002103","-0,002182","0,000716","-0,198355","-0,147162","-0,461694","-0,520696","-0,181949","-0,194439","-0,186522","-0,178327","-0,183164","-0,177253","-0,140733","-0,193724","-0,172226","-0,177797","-0,185592","-0,062281","-0,069518","0,000989","0,000790","-0,058114","0,001807","0,112297"
exercise_hours_per_week,"-0,000848","-0,001245","1,000000","0,001461","0,000083","-0,000308","0,000462","0,000224","-0,000585","0,000362","-0,001020","-0,001216","0,002176","-0,000633","-0,001474","-0,000873","0,000439","-0,000376","0,001080","0,000875","0,000758","0,000462","0,000467","-0,000426","0,000206","0,001247","0,000376","0,000211","-0,000373","0,000666","0,000289","-0,001275","-0,001169","0,000796","0,000028","0,002395","0,000032","-0,001907"
sleep_hours_per_night,"0,000513","0,001742","0,001461","1,000000","-0,000881","-0,000540","0,000425","0,001015","-0,000145","-0,000540","-0,000981","0,001001","-0,000736","-0,000042","-0,000837","0,000641","-0,000795","-0,000985","-0,000388","-0,001652","-0,000753","-0,000605","-0,000326","-0,000243","-0,000269","-0,000884","-0,001897","-0,000629","-0,001617","-0,001370","-0,000707","-0,000215","-0,000744","0,000902","-0,000998","0,000820","-0,000769","-0,000070"
perceived_stress_score,"-0,000993","-0,000561","0,000083","-0,000881","1,000000","0,000187","0,001441","0,000110","0,000201","-0,000076","0,000872","0,000300","-0,000128","-0,000257","-0,000445","-0,000293","0,834351","0,624127","0,376826","0,674719","0,813881","0,818492","0,787228","0,749826","0,769421","0,744556","0,592833","0,813425","0,723960","0,749131","0,779347","0,049720","0,057051","0,000005","-0,000110","0,159238","-0,000684","-0,435999"
self_reported_happiness,"0,001430","0,000046","-0,000308","-0,000540","0,000187","1,000000","0,000516","-0,000244","0,000368","-0,000017","-0,001976","-0,000502","0,000633","-0,000496","0,000202","0,000330","-0,372625","-0,277639","-0,133314","-0,295536","-0,344293","-0,365501","-0,351356","-0,334792","-0,343534","-0,331863","-0,264614","-0,363294","-0,323157","-0,334416","-0,348496","-0,017464","-0,019978","-0,000196","0,001166","-0,106418","-0,001203","0,268591"
body_mass_index,"0,000295","0,001240","0,000462","0,000425","0,001441","0,000516","1,000000","-0,000402","-0,001665","-0,001166","0,000001","0,000398","0,001312","-0,001720","-0,000765","0,001708","0,000690","-0,000115","0,001083","-0,000434","0,000927","0,000546","0,000604","0,000527","0,000474","0,000450","0,000022","0,001209","0,001095","0,000876","0,000764","0,000143","0,000614","0,000346","-0,000556","0,000376","-0,000881","-0,000085"
bloo

column,daily_active_minutes_instagram,likes_given_per_day,time_on_feed_per_day
self_reported_happiness,"-0,372625","-0,365501","-0,363294"


Таким образом, `daily_active_minutes_instagram` (активное время в соцсети), `likes_given_per_day` (поставленные лайки) и `time_on_feed_per_day` (время в новостной ленте) имеют заметную отрицательную корреляцию c уровнем счастья пользователя.
<br/>Визуализируем на графике точки и линейную регрессию:
$$\hat{\beta} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

In [2]:
import org.jetbrains.kotlinx.kandy.ir.Plot
import org.jetbrains.kotlinx.kandy.util.color.Color

// draw linear regression
/**
 * Columns must be [Int]
 */
fun displayLinearRegression(
    columnXName: String, columnXTitle: String,
    columnYName: String, columnYTitle: String,
    limit: Int = learnData.rowsCount()
) { // y = kx + b
    val x = learnData[columnXName].cast<Int>().toList()
    val y = learnData[columnYName].cast<Int>().toList()

    val meanX = x.average()
    val meanY = y.average()

    val k = x.zip(y).sumOf { (xi, yi) ->
        (xi - meanX) * (yi - meanY)
    } / x.sumOf {
        (it - meanX).pow(2)
    }
    val b = meanY - k * meanX

    val xLine = listOf(x.min().toDouble(), x.max().toDouble())
    val yLine = xLine.map { k * it + b }

    DISPLAY(learnData.head(limit).plot {
        points {
            x(column<Double>(columnXName)) {
                axis.name = columnXTitle
            }
            y(column<Int>(columnYName)) {
                axis.name = columnYTitle
            }
            color = Color.BLUE
        }
        line {
            x(xLine)
            y(yLine)
            color = Color.RED
        }
    })
}

displayLinearRegression(
    "daily_active_minutes_instagram", "Ежедневное использование (мин)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)
displayLinearRegression(
    "likes_given_per_day", "Количество поставленных лайков (в день)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)
displayLinearRegression(
    "time_on_feed_per_day", "Ежедневный скролл ленты (мин)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="cKoSuN"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"self_reported_happiness":[8.0,1.0,10.0,1.0,1.0,3.0,10.0,3.0,6.0,10.0,3.0,7.0,10.0,8.0,3.0,10.0,9.0,2.0,3.0,5.0,5.0,8.0,1.0,4.0,10.0,2.0,8.0,9.0,1.0,7.0,2.0,1.0,9.0,8.0,6.0,5.0,7.0,2.0,9.0,6.0,7.0,6.0,4.0,7.0,2.0,1.0,1.0,9.0,6.0,1.0,8.0,6.0,4.0,1.0,5.0,9.0,4.0,9.0,4.0,10.0,8.0,4.0,2.0,1.0,7.0,6.0,8.0,4.0,3.0,8.0,10.0,5.0,2.0,2.0,4.0,7.0,3.0,1.0,6.0,8.0,2.0,4.0,2.0,5.0,5.0,6.0,7.0,4.0,9.0,9.0,1.0,3.0,4.0,6.0,7.0,2.0,5.0,7.0,4.0,4.0,3.0,4.0,6.0,1.0,7.0,10.0,8.0,10.0,8.0,8.0,9.0,6.0,1.0,7.0,6.0,3.0,1.0,10.0,4.0,9.0,7.0,1.0,4.0,10.0,2.0,9.0,9.0,3.0,8.0,8.0,3.0,6.0,1.0,3.0,10.0,4.0,1.0,1.0,10.0,2.0,4.0,5.0,10.0,10.0,8.0,4.0,6.0,5.0,3.0,3.0,2.0,6.0,1.0,7.0,7.0,8.0,6.0,3.0,6.0,3.0,3.0,9.0,6.0,1.0,2.0,9.0,9.0,5.0,9.0,2.0,8.0,4.0,2.0,5.0,1.0,8.0,1.0,9.0,8.0,7.0,5.0,1.0,1.0,9.0,9.0,1.0,8.0,1.0,10.0,6.0,7.0,7.0,7.0,9.0,7.0,10.0,5.0,7.0,9.0,9.0,4.0,3.0,3.0,10.0,3.0,5.0,2.0,9.0,8.0,6.0,3.0,5.0,10.0,8.0,4.0,8.0,9.0,5.0,3.0,2.0,6.0,5.0,2.0,9.0,7.0,6.0,6.0,1.0,10.0,7.0,8.0,6.0,9.0,8.0,2.0,3.0,8.0,6.0,7.0,5.0,7.0,6.0,5.0,5.0,7.0,6.0,3.0,10.0,3.0,8.0,10.0,7.0,5.0,7.0,2.0,1.0,3.0,3.0,4.0,7.0,5.0,3.0,4.0,4.0,4.0,7.0,8.0,7.0,9.0,1.0,4.0,2.0,2.0,7.0,9.0,7.0,9.0,2.0,1.0,6.0,9.0,1.0,2.0,7.0,6.0,2.0,10.0,10.0,9.0,10.0,9.0,6.0,4.0,3.0,5.0,5.0,1.0,1.0,2.0,6.0,9.0,3.0,9.0,10.0,6.0,9.0,6.0,9.0,10.0,5.0,6.0,1.0,6.0,5.0,4.0,9.0,2.0,7.0,7.0,10.0,1.0,6.0,10.0,6.0,2.0,1.0,1.0,1.0,4.0,5.0,2.0,8.0,1.0,2.0,7.0,2.0,5.0,1.0,3.0,2.0,9.0,10.0,4.0,9.0,9.0,8.0,3.0,6.0,4.0,5.0,1.0,6.0,10.0,7.0,6.0,10.0,8.0,3.0,8.0,7.0,6.0,3.0,9.0,3.0,9.0,10.0,9.0,9.0,5.0,2.0,8.0,3.0,1.0,8.0,7.0,7.0,1.0,5.0,1.0,4.0,2.0,2.0,5.0,2.0,6.0,3.0,3.0,7.0,8.0,7.0,8.0,7.0,1.0,2.0,1.0,5.0,8.0,2.0,7.0,8.0,4.0,7.0,6.0,6.0,3.0,10.0,10.0,10.0,6.0,2.0,9.0,8.0,7.0,5.0,9.0,6.0,9.0,3.0,9.0,9.0,8.0,6.0,1.0,1.0,7.0,3.0,4.0,3.0,10.0,6.0,2.0,6.0,9.0,8.0,10.0,2.0,4.0,9.0,8.0,8.0,5.0,6.0,7.0,5.0,9.0,1.0,9.0,10.0,5.0,8.0,4.0,7.0,4.0,4.0,10.0,6.0,4.0,7.0,5.0,4.0,7.0,1.0,3.0,9.0,2.0,4.0,10.0,3.0,5.0,2.0,9.0,1.0,4.0,5.0,6.0,2.0,6.0,2.0,4.0,7.0,1.0,6.0,6.0,8.0,6.0,6.0,2.0,2.0,3.0,8.0,7.0,9.0,10.0,10.0,4.0,8.0,10.0,1.0,5.0,5.0,8.0,5.0,10.0,7.0,7.0,6.0,6.0,3.0,3.0,5.0,6.0,9.0,5.0,7.0,6.0,6.0,1.0,7.0,8.0,5.0,8.0,1.0,1.0,1.0,2.0,1.0,8.0,7.0,9.0,1.0,3.0,5.0,2.0,3.0,2.0,3.0,1.0,8.0,3.0,6.0,6.0,7.0,6.0,6.0,2.0,4.0,4.0,7.0,5.0,9.0,2.0,9.0,8.0,5.0,2.0,8.0,4.0,6.0,7.0,7.0,10.0,9.0,7.0,9.0,9.0,7.0,2.0,9.0,7.0,8.0,10.0,10.0,3.0,2.0,2.0,1.0,5.0,2.0,9.0,1.0,8.0,5.0,4.0,10.0,2.0,8.0,2.0,4.0,1.0,2.0,5.0,2.0,10.0,3.0,9.0,5.0,9.0,8.0,5.0,4.0,6.0,1.0,4.0,10.0,7.0,6.0,4.0,3.0,10.0,1.0,4.0,4.0,9.0,3.0,7.0,5.0,4.0,9.0,2.0,6.0,6.0,3.0,10.0,1.0,7.0,8.0,9.0,1.0,4.0,10.0,2.0,4.0,9.0,3.0,6.0,6.0,3.0,3.0,2.0,8.0,5.0,1.0,4.0,4.0,9.0,5.0,9.0,8.0,8.0,7.0,2.0,7.0,5.0,5.0,5.0,1.0,3.0,10.0,1.0,7.0,3.0,4.0,2.0,4.0,6.0,2.0,3.0,6.0,3.0,6.0,4.0,1.0,1.0,7.0,2.0,1.0,9.0,2.0,7.0,4.0,4.0,4.0,2.0,9.0,10.0,6.0,4.0,2.0,9.0,9.0,2.0,9.0,3.0,5.0,9.0,3.0,7.0,9.0,5.0,9.0,10.0,1.0,4.0,10.0,10.0,9.0,9.0,7.0,6.0,7.0,2.0,10.0,4.0,8.0,7.0,3.0,3.0,2.0,4.0,8.0,7.0,6.0,6.0,2.0,7.0,5.0,8.0,10.0,10.0,9.0,4.0,5.0,8.0,7.0,7.0,6.0,3.0,9.0,9.0,1.0,6.0,4.0,8.0,2.0,9.0,8.0,4.0,5.0,1.0,10.0,3.0,2.0,1.0,10.0,4.0,5.0,6.0,1.0,9.0,4.0,9.0,8.0,8.0,2.0,3.0,1.0,4.0,10.0,4.0,3.0,4.0,2.0,9.0,1.0,8.0,8.0,1.0,5.0,6.0,6.0,3.0,3.0,7.0,10.0,6.0,1.0,4.0,9.0,9.0,4.0,7.0,4.0,3.0,5.0,1.0,2.0,7.0,6.0,2.0,4.0,4.0,2.0,5.0,4.0,1.0,3.0,3.0,9.0,4.0,6.0,3.0,5.0,7.0,5.0,3.0,8.0,9.0,2.0,9.0,2.0,3.0,6.0,1.0,8.0,6.0,6.0,9.0,2.0,8.0,9.0,6.0,9.0,8.0,8.0,9.0,3.0,1.0,10.0,9.0,4.0,4.0,8.0,4.0,1.0,3.0,4.0,2.0,4.0,3.0,9.0,10.0,4.0,8.0,6.0,8.0,5.0,6.0,5.0,10.0,7.0,2.0,1.0,8.0,2.0,6.0,7.0

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="NR5UCk"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"self_reported_happiness":[8.0,1.0,10.0,1.0,1.0,3.0,10.0,3.0,6.0,10.0,3.0,7.0,10.0,8.0,3.0,10.0,9.0,2.0,3.0,5.0,5.0,8.0,1.0,4.0,10.0,2.0,8.0,9.0,1.0,7.0,2.0,1.0,9.0,8.0,6.0,5.0,7.0,2.0,9.0,6.0,7.0,6.0,4.0,7.0,2.0,1.0,1.0,9.0,6.0,1.0,8.0,6.0,4.0,1.0,5.0,9.0,4.0,9.0,4.0,10.0,8.0,4.0,2.0,1.0,7.0,6.0,8.0,4.0,3.0,8.0,10.0,5.0,2.0,2.0,4.0,7.0,3.0,1.0,6.0,8.0,2.0,4.0,2.0,5.0,5.0,6.0,7.0,4.0,9.0,9.0,1.0,3.0,4.0,6.0,7.0,2.0,5.0,7.0,4.0,4.0,3.0,4.0,6.0,1.0,7.0,10.0,8.0,10.0,8.0,8.0,9.0,6.0,1.0,7.0,6.0,3.0,1.0,10.0,4.0,9.0,7.0,1.0,4.0,10.0,2.0,9.0,9.0,3.0,8.0,8.0,3.0,6.0,1.0,3.0,10.0,4.0,1.0,1.0,10.0,2.0,4.0,5.0,10.0,10.0,8.0,4.0,6.0,5.0,3.0,3.0,2.0,6.0,1.0,7.0,7.0,8.0,6.0,3.0,6.0,3.0,3.0,9.0,6.0,1.0,2.0,9.0,9.0,5.0,9.0,2.0,8.0,4.0,2.0,5.0,1.0,8.0,1.0,9.0,8.0,7.0,5.0,1.0,1.0,9.0,9.0,1.0,8.0,1.0,10.0,6.0,7.0,7.0,7.0,9.0,7.0,10.0,5.0,7.0,9.0,9.0,4.0,3.0,3.0,10.0,3.0,5.0,2.0,9.0,8.0,6.0,3.0,5.0,10.0,8.0,4.0,8.0,9.0,5.0,3.0,2.0,6.0,5.0,2.0,9.0,7.0,6.0,6.0,1.0,10.0,7.0,8.0,6.0,9.0,8.0,2.0,3.0,8.0,6.0,7.0,5.0,7.0,6.0,5.0,5.0,7.0,6.0,3.0,10.0,3.0,8.0,10.0,7.0,5.0,7.0,2.0,1.0,3.0,3.0,4.0,7.0,5.0,3.0,4.0,4.0,4.0,7.0,8.0,7.0,9.0,1.0,4.0,2.0,2.0,7.0,9.0,7.0,9.0,2.0,1.0,6.0,9.0,1.0,2.0,7.0,6.0,2.0,10.0,10.0,9.0,10.0,9.0,6.0,4.0,3.0,5.0,5.0,1.0,1.0,2.0,6.0,9.0,3.0,9.0,10.0,6.0,9.0,6.0,9.0,10.0,5.0,6.0,1.0,6.0,5.0,4.0,9.0,2.0,7.0,7.0,10.0,1.0,6.0,10.0,6.0,2.0,1.0,1.0,1.0,4.0,5.0,2.0,8.0,1.0,2.0,7.0,2.0,5.0,1.0,3.0,2.0,9.0,10.0,4.0,9.0,9.0,8.0,3.0,6.0,4.0,5.0,1.0,6.0,10.0,7.0,6.0,10.0,8.0,3.0,8.0,7.0,6.0,3.0,9.0,3.0,9.0,10.0,9.0,9.0,5.0,2.0,8.0,3.0,1.0,8.0,7.0,7.0,1.0,5.0,1.0,4.0,2.0,2.0,5.0,2.0,6.0,3.0,3.0,7.0,8.0,7.0,8.0,7.0,1.0,2.0,1.0,5.0,8.0,2.0,7.0,8.0,4.0,7.0,6.0,6.0,3.0,10.0,10.0,10.0,6.0,2.0,9.0,8.0,7.0,5.0,9.0,6.0,9.0,3.0,9.0,9.0,8.0,6.0,1.0,1.0,7.0,3.0,4.0,3.0,10.0,6.0,2.0,6.0,9.0,8.0,10.0,2.0,4.0,9.0,8.0,8.0,5.0,6.0,7.0,5.0,9.0,1.0,9.0,10.0,5.0,8.0,4.0,7.0,4.0,4.0,10.0,6.0,4.0,7.0,5.0,4.0,7.0,1.0,3.0,9.0,2.0,4.0,10.0,3.0,5.0,2.0,9.0,1.0,4.0,5.0,6.0,2.0,6.0,2.0,4.0,7.0,1.0,6.0,6.0,8.0,6.0,6.0,2.0,2.0,3.0,8.0,7.0,9.0,10.0,10.0,4.0,8.0,10.0,1.0,5.0,5.0,8.0,5.0,10.0,7.0,7.0,6.0,6.0,3.0,3.0,5.0,6.0,9.0,5.0,7.0,6.0,6.0,1.0,7.0,8.0,5.0,8.0,1.0,1.0,1.0,2.0,1.0,8.0,7.0,9.0,1.0,3.0,5.0,2.0,3.0,2.0,3.0,1.0,8.0,3.0,6.0,6.0,7.0,6.0,6.0,2.0,4.0,4.0,7.0,5.0,9.0,2.0,9.0,8.0,5.0,2.0,8.0,4.0,6.0,7.0,7.0,10.0,9.0,7.0,9.0,9.0,7.0,2.0,9.0,7.0,8.0,10.0,10.0,3.0,2.0,2.0,1.0,5.0,2.0,9.0,1.0,8.0,5.0,4.0,10.0,2.0,8.0,2.0,4.0,1.0,2.0,5.0,2.0,10.0,3.0,9.0,5.0,9.0,8.0,5.0,4.0,6.0,1.0,4.0,10.0,7.0,6.0,4.0,3.0,10.0,1.0,4.0,4.0,9.0,3.0,7.0,5.0,4.0,9.0,2.0,6.0,6.0,3.0,10.0,1.0,7.0,8.0,9.0,1.0,4.0,10.0,2.0,4.0,9.0,3.0,6.0,6.0,3.0,3.0,2.0,8.0,5.0,1.0,4.0,4.0,9.0,5.0,9.0,8.0,8.0,7.0,2.0,7.0,5.0,5.0,5.0,1.0,3.0,10.0,1.0,7.0,3.0,4.0,2.0,4.0,6.0,2.0,3.0,6.0,3.0,6.0,4.0,1.0,1.0,7.0,2.0,1.0,9.0,2.0,7.0,4.0,4.0,4.0,2.0,9.0,10.0,6.0,4.0,2.0,9.0,9.0,2.0,9.0,3.0,5.0,9.0,3.0,7.0,9.0,5.0,9.0,10.0,1.0,4.0,10.0,10.0,9.0,9.0,7.0,6.0,7.0,2.0,10.0,4.0,8.0,7.0,3.0,3.0,2.0,4.0,8.0,7.0,6.0,6.0,2.0,7.0,5.0,8.0,10.0,10.0,9.0,4.0,5.0,8.0,7.0,7.0,6.0,3.0,9.0,9.0,1.0,6.0,4.0,8.0,2.0,9.0,8.0,4.0,5.0,1.0,10.0,3.0,2.0,1.0,10.0,4.0,5.0,6.0,1.0,9.0,4.0,9.0,8.0,8.0,2.0,3.0,1.0,4.0,10.0,4.0,3.0,4.0,2.0,9.0,1.0,8.0,8.0,1.0,5.0,6.0,6.0,3.0,3.0,7.0,10.0,6.0,1.0,4.0,9.0,9.0,4.0,7.0,4.0,3.0,5.0,1.0,2.0,7.0,6.0,2.0,4.0,4.0,2.0,5.0,4.0,1.0,3.0,3.0,9.0,4.0,6.0,3.0,5.0,7.0,5.0,3.0,8.0,9.0,2.0,9.0,2.0,3.0,6.0,1.0,8.0,6.0,6.0,9.0,2.0,8.0,9.0,6.0,9.0,8.0,8.0,9.0,3.0,1.0,10.0,9.0,4.0,4.0,8.0,4.0,1.0,3.0,4.0,2.0,4.0,3.0,9.0,10.0,4.0,8.0,6.0,8.0,5.0,6.0,5.0,10.0,7.0,2.0,1.0,8.0,2.0,6.0,7.0

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="1tbAHi"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"time_on_feed_per_day":[2.0,31.0,3.0,108.0,78.0,29.0,64.0,167.0,64.0,115.0,113.0,68.0,3.0,2.0,172.0,3.0,157.0,256.0,102.0,43.0,83.0,2.0,81.0,107.0,26.0,121.0,132.0,3.0,160.0,112.0,36.0,140.0,47.0,81.0,39.0,73.0,16.0,87.0,19.0,3.0,102.0,39.0,140.0,108.0,3.0,176.0,77.0,13.0,102.0,243.0,121.0,174.0,148.0,97.0,143.0,98.0,50.0,12.0,102.0,44.0,96.0,161.0,118.0,69.0,37.0,2.0,130.0,120.0,119.0,48.0,91.0,67.0,155.0,175.0,94.0,68.0,167.0,128.0,192.0,45.0,175.0,55.0,225.0,2.0,42.0,102.0,9.0,19.0,123.0,19.0,78.0,142.0,90.0,2.0,82.0,109.0,3.0,122.0,57.0,100.0,124.0,170.0,87.0,215.0,140.0,108.0,14.0,38.0,161.0,107.0,68.0,119.0,153.0,171.0,124.0,195.0,78.0,105.0,55.0,63.0,163.0,124.0,51.0,32.0,72.0,64.0,60.0,127.0,126.0,29.0,137.0,124.0,129.0,188.0,64.0,131.0,148.0,184.0,66.0,55.0,35.0,132.0,72.0,89.0,103.0,24.0,99.0,82.0,132.0,67.0,124.0,135.0,105.0,161.0,173.0,115.0,121.0,70.0,6.0,142.0,27.0,7.0,77.0,147.0,86.0,59.0,56.0,154.0,94.0,17.0,100.0,164.0,107.0,45.0,43.0,2.0,123.0,69.0,143.0,50.0,87.0,136.0,187.0,177.0,3.0,27.0,38.0,160.0,73.0,75.0,75.0,63.0,126.0,155.0,182.0,127.0,46.0,136.0,3.0,61.0,110.0,111.0,241.0,37.0,101.0,124.0,192.0,153.0,151.0,48.0,98.0,82.0,10.0,135.0,140.0,108.0,121.0,187.0,164.0,133.0,2.0,81.0,101.0,3.0,64.0,139.0,68.0,173.0,103.0,22.0,75.0,110.0,2.0,39.0,126.0,207.0,113.0,2.0,166.0,84.0,167.0,26.0,179.0,22.0,3.0,92.0,163.0,76.0,139.0,143.0,90.0,45.0,19.0,88.0,4.0,106.0,137.0,155.0,67.0,97.0,196.0,90.0,107.0,43.0,69.0,36.0,21.0,122.0,74.0,135.0,167.0,192.0,73.0,179.0,87.0,93.0,163.0,130.0,168.0,3.0,123.0,149.0,181.0,107.0,152.0,39.0,156.0,18.0,3.0,30.0,51.0,118.0,32.0,86.0,89.0,92.0,89.0,83.0,150.0,3.0,99.0,147.0,3.0,3.0,65.0,33.0,187.0,150.0,2.0,112.0,104.0,63.0,64.0,121.0,58.0,11.0,57.0,112.0,61.0,121.0,59.0,98.0,78.0,32.0,24.0,81.0,106.0,81.0,205.0,59.0,127.0,2.0,89.0,54.0,89.0,133.0,44.0,173.0,248.0,149.0,50.0,52.0,68.0,124.0,24.0,3.0,210.0,153.0,74.0,134.0,255.0,50.0,70.0,50.0,82.0,32.0,197.0,83.0,188.0,2.0,68.0,35.0,25.0,107.0,11.0,89.0,140.0,131.0,156.0,27.0,141.0,89.0,59.0,147.0,143.0,28.0,100.0,71.0,128.0,22.0,48.0,86.0,33.0,224.0,17.0,46.0,166.0,2.0,104.0,145.0,148.0,78.0,217.0,169.0,171.0,66.0,32.0,140.0,123.0,109.0,155.0,2.0,85.0,203.0,194.0,7.0,100.0,8.0,120.0,111.0,29.0,32.0,108.0,30.0,130.0,72.0,147.0,107.0,170.0,3.0,116.0,133.0,139.0,158.0,78.0,24.0,71.0,73.0,134.0,171.0,54.0,14.0,148.0,32.0,146.0,118.0,126.0,2.0,60.0,54.0,160.0,126.0,101.0,137.0,78.0,107.0,37.0,3.0,92.0,96.0,53.0,53.0,198.0,123.0,47.0,47.0,18.0,87.0,148.0,57.0,65.0,138.0,124.0,18.0,55.0,116.0,108.0,170.0,236.0,62.0,2.0,185.0,99.0,24.0,109.0,57.0,66.0,66.0,146.0,75.0,45.0,22.0,135.0,140.0,161.0,154.0,16.0,88.0,127.0,60.0,3.0,59.0,51.0,29.0,171.0,85.0,21.0,268.0,139.0,158.0,43.0,95.0,3.0,115.0,49.0,14.0,3.0,186.0,52.0,65.0,2.0,78.0,84.0,79.0,127.0,104.0,103.0,88.0,2.0,152.0,149.0,108.0,167.0,228.0,197.0,63.0,102.0,156.0,84.0,87.0,172.0,25.0,157.0,46.0,162.0,115.0,107.0,21.0,104.0,56.0,101.0,113.0,72.0,29.0,130.0,209.0,171.0,120.0,142.0,19.0,49.0,72.0,60.0,12.0,176.0,35.0,176.0,39.0,71.0,129.0,160.0,162.0,122.0,66.0,87.0,106.0,96.0,50.0,108.0,126.0,3.0,23.0,78.0,131.0,130.0,198.0,60.0,96.0,100.0,183.0,120.0,176.0,82.0,98.0,100.0,84.0,116.0,89.0,154.0,129.0,56.0,150.0,25.0,55.0,3.0,43.0,11.0,86.0,91.0,174.0,111.0,154.0,165.0,21.0,38.0,177.0,50.0,111.0,26.0,59.0,184.0,114.0,140.0,92.0,128.0,53.0,139.0,79.0,265.0,93.0,20.0,184.0,21.0,60.0,121.0,81.0,2.0,241.0,100.0,91.0,120.0,213.0,87.0,121.0,3.0,97.0,137.0,132.0,143.0,168.0,25.0,78.0,149.0,135.0,3.0,185.0,158.0,135.0,30.0,84.0,96.0,56.0,46.0,189.0,190.0,204

Распределим параметры равномерно по термам. По заданию требуется определить все 4 функции принадлежности, поэтому для каждого из 4 параметров (3 входных и 1 выходного) используем разные функции.

Для оценки уровня счастья пользователя (выходной параметр) **треугольную**:
$$
% Треугольная (a, b, c — левая граница, вершина, правая граница)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x \leq b \\
\frac{c - x}{c - b}, & b < x < c \\
0, & x \geq c
\end{cases}
$$

Для количества поставленных лайков - **трапецеидальную**:
$$
% Трапецеидальная (a, b, c, d — границы и плато)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x < b \\
1, & b \leq x \leq c \\
\frac{d - x}{d - c}, & c < x < d \\
0, & x \geq d
\end{cases}
$$

Для ежедневного скролла ленты - **параболическую**:
$$
% Параболическая (a, b — границы)
\mu(x) = \begin{cases}
0, & x \leq a \\
1 - \left(\frac{x - b}{b - a}\right)^2, & a < x \leq b \\
1 - \left(\frac{x - b}{c - b}\right)^2, & b < x < c \\
0, & x \geq c
\end{cases}
$$

Для ежедневного использования соцсети используем - **Гаусса**:
$$
% Гауссова (c — центр, σ — ширина)
\mu(x) = e^{-\frac{(x - c)^2}{2\sigma^2}}
$$

In [18]:
/* API */

enum class MembershipFunctionType {
    TRIANGULAR,
    TRAPEZOIDAL,
    PARABOLIC,
    GAUSSIAN
}

data class LinguisticVariable(
    val columnName: String,
    val userFriendlyName: String,
    val membershipFunctionType: MembershipFunctionType,
    val termNames: List<String>
) {
    val minValue: Int
    val maxValue: Int

    init {
        learnData[columnName]
            .cast<Int>()
            .toList()
            .run {
                minValue = min()
                maxValue = max()
            }
    }
}

fun interface TermChartBuilder {
    /**
     * [fromX], [toX] - both inclusive
     */
    fun append(
        // input
        fromX: Double,
        toX: Double,
        termName: String,
        // output
        bufferX: MutableList<Double>,
        bufferY: MutableList<Double>,
        bufferTerm: MutableList<String>
    )
}

fun displayTerms(
    variable: LinguisticVariable,
    builder: TermChartBuilder
) {
    val step = (variable.maxValue - variable.minValue) / (variable.termNames.size.toDouble() - 1) * 2
    val fromX = variable.minValue - step / 2

    val bufferX = ArrayList<Double>()
    val bufferY = ArrayList<Double>()
    val bufferTerm = ArrayList<String>()

    variable.termNames.forEachIndexed { i, termName ->
        builder.append(
            fromX + step / 2 * i, fromX + step * (i / 2.0 + 1), termName,
            bufferX, bufferY, bufferTerm
        )
    }

    DISPLAY(learnData.plot {
        layout.title = variable.userFriendlyName
        x.axis.limits = variable.minValue.toDouble() ..  variable.maxValue.toDouble()
        y.axis.limits = 0 .. 1
        line {
            x(bufferX)
            y(bufferY)
            color(bufferTerm) { legend.name = "Терм" }
        }
    })
}

/* Term functions */

fun appendTriangularTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(fromX, fromX + (toX - fromX) / 2.0, toX)
    bufferY += listOf(0.0, 1.0, 0.0)
    bufferTerm += List(3) { termName }
}

fun appendTrapezoidalTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(
        fromX,
        fromX + (toX - fromX) / 3.0,
        fromX + (toX - fromX) / 3.0 * 2,
        toX
    )
    bufferY += listOf(0.0, 1.0, 1.0, 0.0)
    bufferTerm += List(4) { termName }
}

fun appendParabolicTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map { 1 - ((it - centerX) / (centerX - fromX)).pow(2) }
    bufferTerm += List(100) { termName }
}

fun appendGaussianTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map {
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (it - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        exp(numerator / denominator)
    }
    bufferTerm += List(100) { termName }
}

/* Impl */

val variables: List<LinguisticVariable> = listOf(
    LinguisticVariable(
        userFriendlyName = "Оценка уровня счастья пользователем",
        columnName = "self_reported_happiness",
        termNames = listOf("ужасно", "плохо", "нормально", "хорошо", "замечательно"),
        membershipFunctionType = MembershipFunctionType.TRIANGULAR
    ),

    LinguisticVariable(
        userFriendlyName = "Лайков поставлено (в день)",
        columnName = "likes_given_per_day",
        termNames = listOf("чуть-чуть", "немного", "не очень много", "много", "очень много"),
        membershipFunctionType = MembershipFunctionType.TRAPEZOIDAL
    ),

    LinguisticVariable(
        userFriendlyName = "Ежедневный скролл ленты (мин)",
        columnName = "time_on_feed_per_day",
        termNames = listOf("недолго", "довольно долго", "крайне долго"),
        membershipFunctionType = MembershipFunctionType.PARABOLIC
    ),

    LinguisticVariable(
        userFriendlyName = "Ежедневное использование соцсети (мин)",
        columnName = "daily_active_minutes_instagram",
        termNames = listOf("немного", "много", "слишком много", "так много, что слов нет"),
        membershipFunctionType = MembershipFunctionType.GAUSSIAN
    )
)

variables // display membership functions charts
    .map {
        it to TermChartBuilder(
            when (it.membershipFunctionType) {
                MembershipFunctionType.TRIANGULAR -> ::appendTriangularTerm
                MembershipFunctionType.TRAPEZOIDAL -> ::appendTrapezoidalTerm
                MembershipFunctionType.PARABOLIC -> ::appendParabolicTerm
                MembershipFunctionType.GAUSSIAN -> ::appendGaussianTerm
            }
        )
    }.toMap()
    .forEach(::displayTerms)

DISPLAY( // display table of variables
    dataFrameOf("Переменная", "Столбец", "Минимальное значение", "Максимальное значение")(
        *variables.flatMap {
            listOf(it.userFriendlyName, it.columnName, it.minValue, it.maxValue)
        }.toTypedArray()
    )
)

/* Used only by following code blocks */

val inputVariables: List<LinguisticVariable> = variables.drop(1)
val outputVariable: LinguisticVariable = variables.first()

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="GU32tw"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Оценка уровня счастья пользователем"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[1.0,10.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["ужасно","ужасно","ужасно","плохо","плохо","плохо","нормально","нормально","нормально","хорошо","хорошо","хорошо","замечательно","замечательно","замечательно"],
"x":[-1.25,1.0,3.25,1.0,3.25,5.5,3.25,5.5,7.75,5.5,7.75,10.0,7.75,10.0,12.25],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"134"
};
 var containerDiv = document.getElementById("GU32tw");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Оценка уровня счастья пользователем 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ужасно 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 плохо 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 нормально 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 хорошо 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 замечательно

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JhEIly"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Лайков поставлено (в день)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[8.0,350.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["чуть-чуть","чуть-чуть","чуть-чуть","чуть-чуть","немного","немного","немного","немного","не очень много","не очень много","не очень много","не очень много","много","много","много","много","очень много","очень много","очень много","очень много"],
"x":[-77.5,-20.5,36.5,93.5,8.0,65.0,122.0,179.0,93.5,150.5,207.5,264.5,179.0,236.0,293.0,350.0,264.5,321.5,378.5,435.5],
"y":[0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"137"
};
 var containerDiv = document.getElementById("JhEIly");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Лайков поставлено (в день) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 чуть-чуть 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 не очень много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 очень много

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="UIaZXF"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневный скролл ленты (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[2.0,328.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне до

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="mraCk5"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневное использование соцсети (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[5.0,580.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","немного","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком много","слишком мног

Переменная,Столбец,Минимальное значение,Максимальное значение
Оценка уровня счастья пользователем,self_reported_happiness,1,10
Лайков поставлено (в день),likes_given_per_day,8,350
Ежедневный скролл ленты (мин),time_on_feed_per_day,2,328
Ежедневное использование соцсети (мин),daily_active_minutes_instagram,5,580


Определим функции принадлежности, соответствующие графикам каждого терма, и определим некоторый программный интерфейс над ними:

In [21]:
/* API */

fun interface MembershipFunction {
    operator fun invoke(x: Double): Double
}

infix fun MembershipFunction.fuzzyOr(function: MembershipFunction) = MembershipFunction { max(this(it), function(it)) }
infix fun MembershipFunction.fuzzyAnd(function: MembershipFunction) = MembershipFunction { min(this(it), function(it)) }

/* Membership functions */

fun interface MembershipFunctionBuilder {
    operator fun invoke(
        fromX: Double,
        toX: Double
    ): MembershipFunction
}

class TriangularMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return when {
            x <= fromX -> 0.0
            x <= centerX -> (x - fromX) / (centerX - fromX)
            x < toX -> (toX - x) / (toX - centerX)
            else -> 0.0 // x >= toX
        }
    }
}

class TrapezoidalMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val thirdX = fromX + (toX - fromX) / 3.0
        val twoThirdX = fromX + (toX - fromX) / 3.0 * 2
        return when {
            x <= fromX -> 0.0
            x < thirdX -> (x - fromX) / (thirdX - fromX)
            x <= twoThirdX -> 1.0
            x < toX -> (toX - x) / (toX - twoThirdX)
            else -> 0.0 // x >= toX
        }
    }
}

class ParabolicMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        return 1 - ((x - centerX) / (centerX - fromX)).pow(2)
    }
}

class GaussianMembershipFunction(
    private val fromX: Double,
    private val toX: Double
) : MembershipFunction {
    override fun invoke(x: Double): Double {
        val centerX = fromX + (toX - fromX) / 2.0
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (x - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        return exp(numerator / denominator)
    }
}

fun buildEvenTerms( // distribute terms from min to max evenly
    termsCount: Int,
    minValue: Int, maxValue: Int,
    builder: MembershipFunctionBuilder
): List<MembershipFunction> {
    val step = (maxValue - minValue) / (termsCount.toDouble() - 1) * 2
    val fromX = minValue - step / 2

    return List(termsCount) { i ->
        builder(
            fromX + step / 2 * i,
            fromX + step * (i / 2 + 1)
        )
    }
}

/* Used only by following code blocks */

private val columnNameToTermNameToMembershipFunction: Map<String, Map<String, MembershipFunction>> =
    variables.map { variable ->
        val termsMemFuns: List<MembershipFunction> = buildEvenTerms(
            termsCount = variable.termNames.size,
            minValue = variable.minValue,
            maxValue = variable.maxValue,
            builder = MembershipFunctionBuilder(
                when (variable.membershipFunctionType) {
                    MembershipFunctionType.TRIANGULAR -> ::TriangularMembershipFunction
                    MembershipFunctionType.TRAPEZOIDAL -> ::TrapezoidalMembershipFunction
                    MembershipFunctionType.PARABOLIC -> ::ParabolicMembershipFunction
                    MembershipFunctionType.GAUSSIAN -> ::GaussianMembershipFunction
                }
            )
        )
        val termNameToFunction: Map<String, MembershipFunction> =
            variable.termNames
                .zip(termsMemFuns)
                .toMap()

        variable.columnName to termNameToFunction
    }.toMap()

operator fun LinguisticVariable.get(termName: String): MembershipFunction =
    columnNameToTermNameToMembershipFunction[columnName]!![termName]!!

Прежде чем реализовать машину нечёткого вывода Мамдани, определим набор правил:

In [ ]:
import java.util.HashMap

typealias InputTerms = Map<String, String> // variable to term
data class ExpectedOutput(val confidenceDegree: Double, val outputTerm: String)

// input terms (variable to term) to output term
private val intermediateRules: MutableMap<InputTerms, ExpectedOutput> = HashMap()

learnData.forEach { row ->
    data class TermAndValue(val termName: String, val memFunValue: Double)

    // variable to term
    val termsWithMaxMembership: Map<LinguisticVariable, TermAndValue> = variables.map { variable ->
        val value: Double = (row[variable.columnName] as Int).toDouble()
        val termWithMaxMembership: TermAndValue = variable.termNames.asSequence()
            .map { termName ->
                TermAndValue(
                    termName = termName,
                    memFunValue = variable[termName](value)
                )
            }.maxBy(TermAndValue::memFunValue)
        variable to termWithMaxMembership
    }.toMap()

    val inputTerms: InputTerms = termsWithMaxMembership.filterKeys { variable ->
        inputVariables.contains(variable)
    }.map { (variable: LinguisticVariable, termAndValue: TermAndValue) ->
        variable.columnName to termAndValue.termName
    }.toMap()

    val confidenceDegree: Double = termsWithMaxMembership.asSequence()
        .map(Map.Entry<*, TermAndValue>::value)
        .map(TermAndValue::memFunValue)
        .reduce(Double::times)

    if (confidenceDegree > intermediateRules[inputTerms]?.confidenceDegree ?: 0)
        intermediateRules[inputTerms] = ExpectedOutput(
            confidenceDegree = confidenceDegree,
            outputTerm = termsWithMaxMembership[outputVariable]!!.termName
        )
}

// TODO print table of rules and update fuzzy output machine code

Далее, реализуем машину нечёткого вывода [Мамдани](https://docs.exponenta.ru/R2021a_nmtnew/fuzzy/types-of-fuzzy-inference-systems.html).
<br/>Алгоритм следующий:
1. Фаззификация входных значений (получить нечёткие значения).
2. Применяем операцию OR (max).
3. Применяем операцию импликации (min)
    - `A => B = not (A and not B)`.
4. Применить операцию агрегации (max).
5. Дефаззифицируем результат (ищем центроид).

Формула координаты `x` центроида (для непрерывного случая):
$$\bar{x} = \frac{\int x \cdot \mu(x) \, dx}{\int \mu(x) \, dx}$$
Используем метод прямоугольников из численного интегрирования:
$$\bar{x} = \frac{\sum_{i=1}^{n} x_i \cdot \mu(x_i)}{\sum_{i=1}^{n} \mu(x_i)}$$

Определим набор правил:
- если
    - "время использования" = "немного"
    - или "время в ленте" = "недолго",
    - тогда "уровень счастья" = "хорошо";
- если
    - "время использования" = "слишком много"
    - или "время в ленте" = "довольно долго"
    - или "лайков поставлено" = "много",
    - тогда "уровень счастья" = "нормально";
- если
    - "время использования" = "так много, что слов нет"
    - или "время в ленте" = "крайне долго"
    - или "лайков поставлено" = "очень много",
    - тогда "уровень счастья" = "плохо".

In [4]:
/* API */

fun interface MembershipFunction {
    operator fun invoke(x: Double): Double
}

infix fun MembershipFunction.fuzzyOr(function: MembershipFunction) = MembershipFunction { max(this(it), function(it)) }
infix fun MembershipFunction.fuzzyAnd(function: MembershipFunction) = MembershipFunction { min(this(it), function(it)) }

data class Term(val name: String, val membership: MembershipFunction)

/**
 *  @param name user-friendly title
 *  @param key original column name in dataset
 */
data class LinguisticVariable(val name: String, val key: String, val exactValue: Double, val terms: List<Term>)
data class Rule(val fuzzyOrTerms: List<Term>, val thenTerm: Term) {
    init { require(fuzzyOrTerms.size > 1) }
}

/* Impl */

fun fuzzifyImplicateAggregate(variables: List<LinguisticVariable>, rules: List<Rule>): MembershipFunction {
    fun Term.linguisticVariable() = variables.first { it.terms.contains(this) }
    operator fun Term.invoke(): Double = membership(linguisticVariable().exactValue)
    operator fun Term.invoke(x: Double): Double = membership(x)

    return rules.asSequence().map { rule ->
        val fuzzifiedInputs: Double = rule.fuzzyOrTerms.maxOf(Term::invoke) // apply fuzzy OR to input terms
        MembershipFunction { x -> min(fuzzifiedInputs, rule.thenTerm(x)) } // apply implication
    }.reduce { f1, f2 -> f1 fuzzyOr f2 } // apply aggregation
}

private val MembershipFunction.centroidX: Double get() {
    val minToMax = learnData["self_reported_happiness"]
        .cast<Int>()
        .toList()
        .run { min() to max() } // min/max values from output param

    val steps = 1000
    val xValues = (0 until steps).map {
        minToMax.first + (minToMax.second - minToMax.first) / steps.toDouble() * it
    }

    val numerator = xValues.asSequence()
        .map { it * this(it) }
        .sum()
    val denominator = xValues.asSequence()
        .map { this(it) }
        .sum()

    return numerator / denominator // find integral using rectangles method
}

/* Mamdani fuzzy output machine solution */

val variables = listOf(
    LinguisticVariable(
        name = "Уровень счастья",
        key = keyHappiness,
        exactValue = 0.0, // ignored, beacuse it's output variable
        terms = buildTerms(
            termNames = termNamesHappiness,
            minToMax = minToMaxHappiness,
            builder = ::TriangularMembershipFunction
        )
    ),
    LinguisticVariable(
        name = "Количество лайков",
        exactValue = 1.0,
        terms = buildTerms(
            termNames = termNamesLikes,
            minToMax = minToMaxLikes,
            builder = ::TrapezoidalMembershipFunction
        )
    ),
    LinguisticVariable(
        name = "Время в ленте",
        exactValue = 60.0,
        terms = buildTerms(
            termNames = termNamesFeed,
            minToMax = minToMaxFeed,
            builder = ::ParabolicMembershipFunction
        )
    ),
    LinguisticVariable(
        name = "Время в соцсети",
        exactValue = 180.0,
        terms = buildTerms(
            termNames = termNamesActiveTime,
            minToMax = minToMaxActiveTime,
            builder = ::GaussianMembershipFunction
        )
    )
)
operator fun List<LinguisticVariable>.get(variableName: String) =
    variables.first() { it.name == variableName }
operator fun LinguisticVariable.get(termName: String) =
    terms.first() { it.name == termName }

val aggregatedFun: MembershipFunction = fuzzifyImplicateAggregate(
    variables = variables,
    rules = listOf(
        Rule(
            fuzzyOrTerms = listOf(
                variables["Время в соцсети"]["немного"],
                variables["Время в ленте"]["недолго"]
            ),
            thenTerm = variables["Уровень счастья"]["хорошо"]
        ),
        Rule(
            fuzzyOrTerms = listOf(
                variables["Время в соцсети"]["слишком много"],
                variables["Время в ленте"]["довольно долго"],
                variables["Количество лайков"]["много"]
            ),
            thenTerm = variables["Уровень счастья"]["нормально"]
        ),
        Rule(
            fuzzyOrTerms = listOf(
                variables["Время в соцсети"]["так много, что слов нет"],
                variables["Время в ленте"]["крайне долго"],
                variables["Количество лайков"]["очень много"]
            ),
            thenTerm = variables["Уровень счастья"]["плохо"]
        )
    )
)
println("Mamdani fuzzy output machine result: ${aggregatedFun.centroidX}")

// visualize aggregated function
val steps = 1000
val xLines: List<Double> = (0 until steps).map {
    minToMaxHappiness.first + (minToMaxHappiness.second - minToMaxHappiness.first) / steps.toDouble() * it
}
val yLines = xLines.map {
    aggregatedFun(it)
}

plot {
    layout.title = "Результат агрегации"
    line {
        x(xLines)
        y(yLines)
    }
}

Mamdani fuzzy output machine result: 5.6911444473759865


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Ns6Un1"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Результат агрегации"
},
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[1.0,1.009,1.018,1.027,1.036,1.045,1.054,1.063,1.072,1.081,1.09,1.099,1.108,1.117,1.126,1.135,1.144,1.153,1.162,1.171,1.18,1.189,1.198,1.207,1.216,1.225,1.234,1.2429999999999999,1.252,1.261,1.27,1.279,1.288,1.297,1.306,1.315,1.3239999999999998,1.333,1.342,1.351,1.3599999999999999,1.369,1.378,1.387,1.396,1.405,1.414,1.423,1.432,1.4409999999999998,1.45,1.459,1.468,1.4769999999999999,1.486,1.4949999999999999,1.504,1.513,1.5219999999999998,1.531,1.54,1.549,1.5579999999999998,1.567,1.576,1.585,1.5939999999999999,1.603,1.612,1.621,1.63,1.6389999999999998,1.648,1.657,1.666,1.6749999999999998,1.684,1.693,1.702,1.7109999999999999,1.72,1.729,1.738,1.7469999999999999,1.7559999999999998,1.765,1.774,1.783,1.7919999999999998,1.801,1.81,1.819,1.8279999999999998,1.837,1.846,1.855,1.8639999999999999,1.8729999999999998,1.882,1.891,1.9,1.9089999999999998,1.918,1.927,1.936,1.9449999999999998,1.954,1.963,1.972,1.9809999999999999,1.9899999999999998,1.9989999999999999,2.008,2.017,2.026,2.035,2.0439999999999996,2.053,2.062,2.0709999999999997,2.08,2.089,2.098,2.107,2.1159999999999997,2.125,2.134,2.143,2.152,2.1609999999999996,2.17,2.179,2.1879999999999997,2.197,2.206,2.215,2.224,2.2329999999999997,2.242,2.251,2.26,2.269,2.2779999999999996,2.287,2.296,2.3049999999999997,2.314,2.323,2.332,2.341,2.3499999999999996,2.359,2.368,2.377,2.386,2.3949999999999996,2.404,2.413,2.4219999999999997,2.431,2.44,2.449,2.458,2.4669999999999996,2.476,2.485,2.4939999999999998,2.503,2.5119999999999996,2.521,2.53,2.5389999999999997,2.548,2.557,2.566,2.575,2.5839999999999996,2.593,2.602,2.6109999999999998,2.62,2.6289999999999996,2.638,2.647,2.6559999999999997,2.665,2.674,2.683,2.692,2.7009999999999996,2.71,2.719,2.7279999999999998,2.737,2.7459999999999996,2.755,2.764,2.7729999999999997,2.782,2.791,2.8,2.809,2.8179999999999996,2.827,2.836,2.8449999999999998,2.854,2.8629999999999995,2.872,2.881,2.8899999999999997,2.899,2.908,2.917,2.926,2.9349999999999996,2.944,2.953,2.9619999999999997,2.971,2.9799999999999995,2.989,2.9979999999999998,3.0069999999999997,3.016,3.025,3.034,3.0429999999999997,3.052,3.061,3.07,3.0789999999999997,3.0879999999999996,3.097,3.106,3.1149999999999998,3.1239999999999997,3.133,3.142,3.151,3.1599999999999997,3.169,3.178,3.187,3.1959999999999997,3.2049999999999996,3.214,3.223,3.2319999999999998,3.2409999999999997,3.25,3.259,3.268,3.2769999999999997,3.286,3.295,3.304,3.3129999999999997,3.3219999999999996,3.331,3.34,3.3489999999999998,3.3579999999999997,3.367,3.376,3.385,3.3939999999999997,3.403,3.412,3.421,3.4299999999999997,3.4389999999999996,3.448,3.457,3.4659999999999997,3.4749999999999996,3.484,3.493,3.502,3.5109999999999997,3.52,3.529,3.538,3.5469999999999997,3.5559999999999996,3.565,3.574,3.5829999999999997,3.5919999999999996,3.601,3.61,3.6189999999999998,3.6279999999999997,3.637,3.646,3.655,3.6639999999999997,3.6729999999999996,3.682,3.691,3.6999999999999997,3.7089999999999996,3.718,3.727,3.7359999999999998,3.7449999999999997,3.754,3.763,3.772,3.7809999999999997,3.7899999999999996,3.799,3.808,3.8169999999999997,3.8259999999999996,3.835,3.844,3.8529999999999998,3.8619999999999997,3.871,3.88,3.889,3.8979999999999997,3.9069999999999996,3.916,3.925,3.9339999999999997,3.9429999999999996,3.952,3.961,3.9699999999999998,3.9789999999999996,3.988,3.997,4.006,4.015,4.023999999999999,4.0329999999999995